# Java Unit Test Generator

Uses ChatGPT to generate Unit Tests for given Java code snippet using JUnit

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import tempfile

In [ ]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [ ]:
# initialize

openai = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

In [ ]:
system_message = (
    "You are an assistant that writes unit tests for Java code using JUnit. "
    "Respond only with JUnit test code. Avoid explanations and minimize comments, using them only when necessary. "
    "Ensure the test suite runs correctly and achieves optimal code coverage."
)

In [ ]:
def user_prompt_for(java_code):
    user_prompt = (
        "Write JUnit tests for this Java code and ensure the test suite runs correctly and achieves optimal code coverage. "
        "Respond only with JUnit test code; do not explain your work other than a few comments. "
        f"{java_code}"
    )
    return user_prompt

In [ ]:
def messages_for(java_code):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(java_code)}
    ]

In [ ]:
def generate_tests(java_code_input, file_input):
    if file_input is not None:
        with open(file_input.name, 'r', encoding='utf-8') as f:
            java_code = f.read()
    elif java_code_input:
        java_code = java_code_input
    else:
        return "Please provide either Java code or upload a file."
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(java_code), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```java\n','').replace('```','')

In [ ]:
def export_to_file(test_code):
    if test_code is None or not test_code.strip():
        return None, gr.update(visible=False)
    with tempfile.NamedTemporaryFile(mode="w+", suffix=".java", delete=False) as f:
        f.write(test_code)
        return f.name, gr.update(visible=True)

In [ ]:
with gr.Blocks() as ui:
    gr.Markdown("# Java JUnit Test Generator")
    
    with gr.Row():
        java_code_input = gr.Textbox(label="Enter Java Code", placeholder="Write or paste your Java code here", lines=15)
    with gr.Row():
        file_input = gr.File(label="Upload Java File", file_types=[".java"], file_count="single")
    with gr.Row():
        submit_button = gr.Button("Generate JUnit Tests")
    with gr.Row():
        output_text = gr.Textbox(label="Generated JUnit Tests", lines=15)
    with gr.Row():
        export_button = gr.Button("Export tests as .java File")
        file_output = gr.File(label="Download", visible=False)

    # Connect streaming generator
    submit_button.click(fn=generate_tests, inputs=[java_code_input, file_input], outputs=output_text)
    export_button.click(fn=export_to_file, inputs=output_text, outputs=[file_output, file_output])

ui.launch(inbrowser=True)